In [ ]:
# Cell 1: Import dependencies and resolve local config paths
from pathlib import Path
import pickle

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import yaml
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import load_model

notebook_dir = Path().resolve()
project_root = notebook_dir.parent if notebook_dir.name == 'notebooks' else notebook_dir

cfg_path = project_root / 'configs' / 'local_data_paths.yaml'
if cfg_path.exists():
    cfg = yaml.safe_load(cfg_path.read_text())
    pkl_path = Path(cfg.get('datasets', {}).get('rml2016', {}).get('pkl', '/scratch/rameyjm7/datasets/RML2016/RML2016.10a_dict.pkl'))
else:
    pkl_path = Path('/scratch/rameyjm7/datasets/RML2016/RML2016.10a_dict.pkl')

model_path = project_root / 'models' / 'rml2016' / 'rml2016_lstm_rnn_2024.keras'
print('RML2016 dataset:', pkl_path)
print('RML2016 model:', model_path)
assert pkl_path.exists(), f'Missing dataset: {pkl_path}'
assert model_path.exists(), f'Missing model: {model_path}'

In [ ]:
# Cell 2: Load dataset and prepare train/test split with IQ+SNR features
with pkl_path.open('rb') as f:
    data = pickle.load(f, encoding='latin1')

X_rows, y_rows = [], []
for (mod, snr), signals in data.items():
    for signal in signals:
        iq = np.vstack([signal[0], signal[1]]).T.astype(np.float32)
        snr_col = np.full((iq.shape[0], 1), snr, dtype=np.float32)
        X_rows.append(np.hstack([iq, snr_col]))
        y_rows.append(mod)

X = np.asarray(X_rows, dtype=np.float32)
y_text = np.asarray(y_rows)

encoder = LabelEncoder()
y = encoder.fit_transform(y_text)

_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('X_test shape:', X_test.shape)
print('Classes:', list(encoder.classes_))

In [ ]:
# Cell 3: Run model inference and print metrics
model = load_model(model_path, compile=False)
y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)

acc = accuracy_score(y_test, y_pred)
print(f'RML2016 evaluation accuracy: {acc:.4f}')
print(classification_report(y_test, y_pred, target_names=encoder.classes_, zero_division=0))

In [ ]:
# Cell 4: Plot confusion matrix for RML2016
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, cmap='Blues', xticklabels=encoder.classes_, yticklabels=encoder.classes_)
plt.title('RML2016 Confusion Matrix')
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()